# Baseline nnU-Net 3D full resolution (Dice + CE)

Entrenamiento de la configuración `3d_fullres` de nnU-Net v2 sobre 200 casos de ImageCAS (conjunto `Dataset501_ImageCAS`), con el fold 0 y 100 épocas. El modelo resultante constituye el baseline de referencia (pérdida Dice + entropía cruzada) frente al que se comparan las variantes con soft-clDice.


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    raise RuntimeError("No hay GPU disponible.")

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.25 GB


In [2]:
!python -m pip install -q nnunetv2
!python -m pip install -q --force-reinstall "numpy==2.0.2"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.3 MB/s eta 0:00

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import numpy as np
import torch
import numba
import nnunetv2

print("NumPy:", np.__version__)
print("Numba:", numba.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("nnU-Net OK")

NumPy: 2.0.2
Numba: 0.60.0
PyTorch: 2.11.0+cu128
CUDA: 12.8
CUDA disponible: True
GPU: NVIDIA A100-SXM4-80GB
nnU-Net OK


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

NNUNET_RAW = BASE / "nnUNet_raw"
NNUNET_PREPROCESSED = BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE / "nnUNet_results"

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/drive/MyDrive/TFM_nnUNet/nnUNet_raw
nnUNet_preprocessed: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed
nnUNet_results: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results


In [4]:
ruta_dataset_preprocessed = (
    NNUNET_PREPROCESSED / "Dataset501_ImageCAS"
)

carpeta_3d = (
    ruta_dataset_preprocessed / "nnUNetPlans_3d_fullres"
)

print("Dataset preprocesado:", ruta_dataset_preprocessed.exists())
print("Configuración 3D FullRes:", carpeta_3d.exists())

if carpeta_3d.exists():
    imagenes = [
        archivo
        for archivo in carpeta_3d.glob("*.b2nd")
        if not archivo.name.endswith("_seg.b2nd")
    ]

    segmentaciones = list(
        carpeta_3d.glob("*_seg.b2nd")
    )

    print("Imágenes preprocesadas:", len(imagenes))
    print("Segmentaciones:", len(segmentaciones))

Dataset preprocesado: True
Configuración 3D FullRes: True
Imágenes preprocesadas: 113
Segmentaciones: 114


In [ ]:
import os
from pathlib import Path

print("RAW:", os.environ["nnUNet_raw"])
print("PREPROCESSED:", os.environ["nnUNet_preprocessed"])
print("RESULTS:", os.environ["nnUNet_results"])

print(
    Path(os.environ["nnUNet_raw"], "Dataset501_ImageCAS").exists()
)

RAW: /content/drive/MyDrive/TFM_nnUNet/nnUNet_raw
PREPROCESSED: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed
RESULTS: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results
True


In [6]:
from pathlib import Path
import os

raw = Path(os.environ["nnUNet_raw"]) / "Dataset501_ImageCAS"

img_0002 = raw / "imagesTr" / "ImageCAS_0002_0000.nii.gz"
seg_0002 = raw / "labelsTr" / "ImageCAS_0002.nii.gz"

print("Imagen:", img_0002.exists(), img_0002)
print("Máscara:", seg_0002.exists(), seg_0002)

Imagen: True /content/drive/MyDrive/TFM_nnUNet/nnUNet_raw/Dataset501_ImageCAS/imagesTr/ImageCAS_0002_0000.nii.gz
Máscara: True /content/drive/MyDrive/TFM_nnUNet/nnUNet_raw/Dataset501_ImageCAS/labelsTr/ImageCAS_0002.nii.gz


In [7]:
from pathlib import Path

ruta = NNUNET_PREPROCESSED / "Dataset501_ImageCAS"

print(ruta)

print("\nContenido:\n")

for f in sorted(ruta.iterdir()):
    print(f.name)

/content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset501_ImageCAS

Contenido:

dataset.json
dataset_fingerprint.json
gt_segmentations
nnUNetPlans.json
nnUNetPlans_2d
nnUNetPlans_3d_fullres
splits_final.json


In [8]:
!nnUNetv2_plan_and_preprocess \
-d 501 \
-c 3d_fullres \
-npfp 2 \
-np 2

Fingerprint extraction...
Dataset501_ImageCAS
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [0.515      0.36210938 0.36210938]. 
Current patch size: (np.int64(96), np.int64(160), np.int64(160)). 
Current median shape: [266.99029126 497.08737864 497.08737864]
Attempting to find 3d_lowres config. 
Current spacing: [0.53045    0.37297266 0.37297266]. 
Current patch size: (np.int64(96), np.int64(160), np.int64(160)). 
Current median shape: [259.21387501 482.60910548 482.60910548]
Attempting to find 3d_lowres config. 
Current spacing: [0.5463635  0.38416184 0.38416184]. 
Current patch size: (np.int64(96), np.int64(160), np.int64(160)). 
Current median shape: [251.66395632 468.552529

In [9]:
ruta_3d = (
    NNUNET_PREPROCESSED
    / "Dataset501_ImageCAS"
    / "nnUNetPlans_3d_fullres"
)

print("Existe 3D FullRes:", ruta_3d.exists())

if ruta_3d.exists():
    imagenes_3d = [
        f for f in ruta_3d.glob("*.b2nd")
        if not f.name.endswith("_seg.b2nd")
    ]

    mascaras_3d = list(
        ruta_3d.glob("*_seg.b2nd")
    )

    print("Imágenes 3D:", len(imagenes_3d))
    print("Máscaras 3D:", len(mascaras_3d))

Existe 3D FullRes: True
Imágenes 3D: 200
Máscaras 3D: 200


In [10]:
from pathlib import Path

ruta = NNUNET_PREPROCESSED / "Dataset501_ImageCAS" / "nnUNetPlans_3d_fullres"

imagenes = sorted([
    f.stem
    for f in ruta.glob("*.b2nd")
    if not f.name.endswith("_seg.b2nd")
])

mascaras = sorted([
    f.stem.replace("_seg", "")
    for f in ruta.glob("*_seg.b2nd")
])

faltan = sorted(set(mascaras) - set(imagenes))

print("Imágenes:", len(imagenes))
print("Máscaras:", len(mascaras))

print("\nCasos sin imagen:")
print(faltan)

Imágenes: 200
Máscaras: 200

Casos sin imagen:
[]


In [11]:
from pathlib import Path
import os

carpeta = (
    Path(os.environ["nnUNet_preprocessed"])
    / "Dataset501_ImageCAS"
    / "nnUNetPlans_3d_fullres"
)

print("Carpeta:", carpeta)
print("¿Existe?", carpeta.exists())

for caso in ["ImageCAS_0001", "ImageCAS_0002", "ImageCAS_0003"]:
    print(f"\n{caso}")

    archivos = [
        carpeta / f"{caso}.b2nd",
        carpeta / f"{caso}_seg.b2nd",
        carpeta / f"{caso}.pkl",
    ]

    for archivo in archivos:
        if archivo.exists():
            print(f"✅ {archivo.name} — {archivo.stat().st_size / 1024**2:.2f} MB")
        else:
            print(f"❌ {archivo.name}")

Carpeta: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset501_ImageCAS/nnUNetPlans_3d_fullres
¿Existe? True

ImageCAS_0001
✅ ImageCAS_0001.b2nd — 177.87 MB
✅ ImageCAS_0001_seg.b2nd — 0.32 MB
✅ ImageCAS_0001.pkl — 0.31 MB

ImageCAS_0002
✅ ImageCAS_0002.b2nd — 188.41 MB
✅ ImageCAS_0002_seg.b2nd — 0.28 MB
✅ ImageCAS_0002.pkl — 0.31 MB

ImageCAS_0003
✅ ImageCAS_0003.b2nd — 147.06 MB
✅ ImageCAS_0003_seg.b2nd — 0.29 MB
✅ ImageCAS_0003.pkl — 0.31 MB


In [12]:
from pathlib import Path
import os

carpeta = Path(os.environ["nnUNet_preprocessed"]) / "Dataset501_ImageCAS" / "nnUNetPlans_3d_fullres"

casos = sorted(set(
    p.name.replace("_seg","").replace(".b2nd","").replace(".pkl","")
    for p in carpeta.iterdir()
))

print("Número de casos encontrados:", len(casos))

for c in casos[:5]:
    print(c)

print("...")

for c in casos[-5:]:
    print(c)

Número de casos encontrados: 200
ImageCAS_0001
ImageCAS_0002
ImageCAS_0003
ImageCAS_0004
ImageCAS_0005
...
ImageCAS_0196
ImageCAS_0197
ImageCAS_0198
ImageCAS_0199
ImageCAS_0200


In [13]:
imagenes = [
    p for p in carpeta.glob("*.b2nd")
    if not p.name.endswith("_seg.b2nd")
]

mascaras = list(carpeta.glob("*_seg.b2nd"))
propiedades = list(carpeta.glob("*.pkl"))

print("Imágenes:", len(imagenes))
print("Máscaras:", len(mascaras))
print("PKL:", len(propiedades))

Imágenes: 200
Máscaras: 200
PKL: 200


In [14]:
!nnUNetv2_train \
501 \
3d_fullres \
0 \
-tr nnUNetTrainer_100epochs \
--npz


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-08-10 16:45:08.081081: Using torch.compile...
2026-08-10 16:45:11.249031: do_dummy_2d_data_aug: False
2026-08-10 16:45:11.261600: Using splits from existing split file: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset501_ImageCAS/splits_final.json
2026-08-10 16:45:11.866667: Th

In [ ]:
from IPython.display import Image, display

fold_dir = folders[0]

progress = fold_dir / "progress.png"

print(progress)
display(Image(filename=str(progress)))